# Final Assignment - Computational Macroeconomics
### Student: Binyam Yohannes Olango
### ID: 4767382
### Date: 03 Aug 2026

---

## Overview
This notebook contains my full solutions to the Final Assignment.  
All notations and formulas follows the assignment sheet.


In [28]:
# Importing packages needed for the tasks
import numpy as np
from scipy.optimize import fsolve

## Task 1


In [29]:
# Defining parameters
sigma= 1
kappa=0.3
beta=0.995
phi_pi=1.5
phi_y=0.1
rho_mu=0.7


def model(z_t, Ez_t1, z_tm1, u_t):
    """
    Computes the residuals of the model equation: A z_t = M E_t[z_{t+1}] + D z_{t-1} + u_t

    Inputs:
    z_t: vector of current-period variables [mu_t, y_t, pi_t, i_t]
    Ez_t1: vector of expected next-period variables E_t[z_{t+1}]
    z_tm1: vector of lagged variables z_{t-1}
    u_t: vector of shocks

    Return: vector of residuals.
    """

    # Contribution of current-period variables
    left = A @ z_t

    # Contribution of expected future variables, lagged variables, and shocks
    right = M @ Ez_t1 + D @ z_tm1 + u_t

    # Residuals: must be zero for the model to hold
    return left - right

# Writing Matrix A which contains the coefficients on current-period variables
A = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 1/sigma],
    [-1, -kappa, 1, 0],
    [0, 0, 0, 1]
], dtype=float)

# Writing Matrix M which contains the coefficients on expected future variables
M = np.array([
    [0, 0, 0, 0],
    [0, 1, 1/sigma, 0],
    [0, 0, beta, 0],
    [0, 0, 0, 0]
], dtype=float)

# Writing Matrix D which contains the coefficients on lagged variables
D = np.array([
    [rho_mu, 0, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 0],
    [0, phi_y, phi_pi, 0]
], dtype=float)


## Task 2

In [30]:
def steady_state(z_guess):
    """
    In steady state: z_t = E_t[z_{t+1}] = z_{t-1} = z_guess, shocks (u_t) = 0

    Input: z_guess - a vector that containing trial values for these four variables: [mu, Y, pi, i]
    
    Return: The difference of implied z_t and z_guessed steady state values
    """
    
    # Creating a NumPy array from the input z_guess
    z_guess = np.array(z_guess, dtype=float)

    
    # With the above assumption calculate the right and left hand side of Equation 6 from the sheet separately
    right = M @ z_guess + D @ z_guess
    left = A @ z_guess

    return left - right

## Task 3


In [31]:
# Create an initial guess for the state vector.
# np.zeros(4) creates a NumPy array with elements value 0
z0 = np.zeros(4)

# fsolve attempts to find values of z such that: steady_state(z) = [0, 0, 0, 0]
z_ss = fsolve(steady_state, z0)


Because the steady state of the model is exactly zero, the MSV solution does not need a constant term. A constant term would only be required if the model converged to a non‑zero steady state. Since fsolve shows that all steady‑state values are zero, the MSV solution can be written entirely as a linear function of lagged variables and shocks, with no additional constant needed to shift the solution.

## Task 4

In [32]:
def LTI(A, M, D, F_guess):
    """
    Applies the Linear Time Iteration (LTI) algorithm to compute
    the matrices F and Q in the MSV solution:  z_t = F z_{t-1} + Q u_t

    Inputs:
    A, M, D = model matrices from Task 1
    F_guess = initial guess for F

    Return:
    F_new = matrix of coefficients on lagged variables
    Q = matrix of coefficients on shocks
    """

    # F_new = (A - M F_guess)^(-1) * D
    F_new = np.linalg.inv(A - M @ F_guess) @ D

    # Q = (A - M F_new)^(-1)
    Q = np.linalg.inv(A - M @ F_new)

    return F_new, Q


## Task 5

In [33]:
def solve_LTI(A, M, D, tol=1e-10, max_iter=500):
    """
    Repeatedly calls the LTI algorithm until F converges.
    
    Inputs:
    A, M, D      → model matrices from task 1
    tol          → convergence tolerance (how close F_new must be to F_old)
    max_iter     → maximum number of iterations allowed
    
    Returns:
    F            → converged matrix of coefficients on lagged variables
    Q            → matrix of coefficients on shocks
    """

    # Step 1: Start with an initial guess for F and Q (zero matrix is standard)
    F_old = np.zeros_like(A)
    Q = np.zeros_like(A)  # initial value

    # Step 2: Iterate the LTI update until convergence
    for iteration in range(max_iter):

        # Call the LTI function from Task 4
        F_new, Q = LTI(A, M, D, F_old)

        # Step 3: Check convergence
        # If the difference between F_new and F_old is tiny, we stop
        if np.max(np.abs(F_new - F_old)) < tol:
            print(f"Converged after {iteration+1} iterations.")
            return F_new, Q

        # Step 4: Update F_old for the next iteration
        F_old = F_new

    # If we reach max_iter without converging, warn the user
    print("Warning: LTI did not converge within max_iter.")
    return F_old, Q


## Task 6


In [34]:
# Calling the solve_LTI function to get the F and Q
F, Q = solve_LTI(A, M, D)

# Columns of F correspond to lagged state variables [mu_{t-1}, y_{t-1}, pi_{t-1}, i_{t-1}]
C_mu   = F[:, 0]
C_y    = F[:, 1]
C_pi   = F[:, 2]

# Columns of Q correspond to components of shocks vector u_t = [eps_mu, 0, 0, eps_i]
C_eps_mu = Q[:, 0]
C_eps_i  = Q[:, 3]


Converged after 37 iterations.


## Task 7

In [43]:
T = 30  # number of periods for the Impulse Response Function

# Arrays to store the paths of mu, Y, pi, i
mu_path = np.zeros(T)
Y_path  = np.zeros(T)
pi_path = np.zeros(T)
i_path  = np.zeros(T)

# Initial state at t = -1 (before the shock): all variables (mu, Y, pi, i) at steady state (zero)
z_tm1 = np.zeros(4)

# Size of the one-time shock to eps_mu_t at t = 0
eps_mu_t = 0.01

for t in range(T):
    if t == 0:
        # At t = 0: one-time shock to eps_mu_t, no shock to eps_i_t
        # u_t = [eps_mu_t, 0, 0, eps_i_t], we got this shocks vector by deriving from the model equation
        u_t = np.array([eps_mu_t, 0.0, 0.0, 0.0])
    else:
        # After t = 0: no more shocks
        u_t = np.zeros(4)

    # Apply MSV solution: z_t = C_mu*mu_{t-1} + C_y*y_{t-1} + C_pi*pi_{t-1} + C_eps_mu*eps_mu_t + C_eps_i*eps_i_t
    z_t = (
        C_mu * z_tm1[0]
        + C_y * z_tm1[1]
        + C_pi * z_tm1[2]
        + C_eps_mu * u_t[0]
        + C_eps_i * u_t[3]
    )

    # Store each variable's path
    mu_path[t] = z_t[0]
    Y_path[t]  = z_t[1]
    pi_path[t] = z_t[2]
    i_path[t]  = z_t[3]

    # Update lagged state for next period
    z_tm1 = z_t

print('The impulse responses to a 0.01 shock to mu_t:')
print(z_tm1)


The impulse responses to a 0.01 shock to mu_t:
[ 3.21990576e-07 -8.18994627e-07  2.51374588e-07  4.21660600e-07]


## Task 8 - [Insert short description from the PDF]
### Objective
Summarize what the task requires (e.g., compute steady state, derive Euler equation, etc.).

### Given
- List the parameters and equations provided in the assignment sheet.

### Approach
Explain briefly how you will solve it.


## Task 9 - [Insert short description from the PDF]
### Objective
Summarize what the task requires (e.g., compute steady state, derive Euler equation, etc.).

### Given
- List the parameters and equations provided in the assignment sheet.

### Approach
Explain briefly how you will solve it.


## Task 10 - [Insert short description from the PDF]
### Objective
Summarize what the task requires (e.g., compute steady state, derive Euler equation, etc.).

### Given
- List the parameters and equations provided in the assignment sheet.

### Approach
Explain briefly how you will solve it.


## Task 11 - [Insert short description from the PDF]
### Objective
Summarize what the task requires (e.g., compute steady state, derive Euler equation, etc.).

### Given
- List the parameters and equations provided in the assignment sheet.

### Approach
Explain briefly how you will solve it.


## Task 12 - [Insert short description from the PDF]
### Objective
Summarize what the task requires (e.g., compute steady state, derive Euler equation, etc.).

### Given
- List the parameters and equations provided in the assignment sheet.

### Approach
Explain briefly how you will solve it.


## Task 13 - [Insert short description from the PDF]
### Objective
Summarize what the task requires (e.g., compute steady state, derive Euler equation, etc.).

### Given
- List the parameters and equations provided in the assignment sheet.

### Approach
Explain briefly how you will solve it.


## Task 14 - [Insert short description from the PDF]
### Objective
Summarize what the task requires (e.g., compute steady state, derive Euler equation, etc.).

### Given
- List the parameters and equations provided in the assignment sheet.

### Approach
Explain briefly how you will solve it.
